<a href="https://colab.research.google.com/github/melisa176/phishing-detection/blob/main/notebooks/02_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Análisis Exploratorio de Datos (EDA)

Este notebook examina el dataset unido sin modificarlo, para entender
qué problemas existen antes de decidir la limpieza en el notebook 03.

Antes de cargar el dataset guardado, se vuelve a generar la columna
`source` (que no se guardó en la versión actual de `01_extraccion.ipynb`)
para poder comparar las 3 fuentes a lo largo de este análisis.

## 0. Instalación de dependencias

In [ ]:
!pip install langdetect matplotlib -q

## 1. Cargar el dataset y agregar la columna 'source'

`source` no quedó guardada en `dataset_unido_raw.csv`, así que la
reconstruimos aquí usando el orden de concatenación original
(pot → nazario → enron) y los conteos conocidos de cada fuente.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

RUTA_DATASET = '/content/drive/MyDrive/dataset_unido_raw.csv'
df = pd.read_csv(RUTA_DATASET)

# Conteos conocidos de la extracción (de 01_extraccion.ipynb)
N_POT = 8614
N_NAZARIO = 3466
N_ENRON = len(df) - N_POT - N_NAZARIO

df['source'] = (
    ['phishing_pot'] * N_POT +
    ['nazario'] * N_NAZARIO +
    ['enron'] * N_ENRON
)

print('Filas:', len(df))
print('Columnas:', df.columns.tolist())
print()
print('Verificación de fuente x etiqueta (debe ser: pot=1, nazario=1, enron=0):')
print(pd.crosstab(df['source'], df['label']))

## 2. Distribución de longitud del texto

Analizamos longitud en caracteres y en palabras. En sesiones de limpieza
anteriores sobre datos de procedencia similar se encontró un patrón de
"salto brusco" en los percentiles altos (95 a 99), señal de filas
infladas con relleno artificial. Verificamos si este dataset muestra el
mismo patrón.

In [ ]:
df['n_caracteres'] = df['text'].astype(str).str.len()
df['n_palabras'] = df['text'].astype(str).str.split().str.len()

print('--- Longitud en caracteres ---')
print(df['n_caracteres'].describe())
print()
print('Percentiles altos (donde suelen aparecer outliers de relleno):')
for p in [90, 95, 97, 99, 99.5, 100]:
    print(f'  p{p}: {df["n_caracteres"].quantile(p/100):.0f}')

In [ ]:
fila_extrema = df.loc[df['n_caracteres'].idxmax()]
print('Fuente:', fila_extrema['source'])
print('Label:', fila_extrema['label'])
print('Longitud:', fila_extrema['n_caracteres'])
print()
print('Primeros 300 caracteres:')
print(fila_extrema['text'][:300])
print()
print('Últimos 200 caracteres:')
print(fila_extrema['text'][-200:])

## 4. Técnica de "hidden text spam" (HTML con texto oculto)

La fila de longitud extrema (2.6M caracteres, fuente phishing_pot, banco
Bradesco) reveló un patrón distinto al relleno random alfanumérico visto
en sesiones anteriores: HTML real con `<span>` ocultos vía
`visibility: hidden` y `font-size: 0`, una técnica clásica de evasión de
filtros de spam (el correo se ve normal visualmente, pero el código
fuente está inflado con texto basura invisible).

**Nota sobre idiomas:** se confirmó en literatura (ChatSpamDetector, 2024)
que datasets de phishing reales de procedencia similar a la nuestra
contienen naturalmente correos en portugués, alemán, holandés, francés,
etc. — dirigidos a marcas y bancos regionales (incluyendo Bradesco,
el mismo banco de este ejemplo). Por lo tanto, el patrón buscado aquí
se basa en sintaxis CSS (`visibility:hidden`, `font-size:0`), NO en
palabras de ningún idioma, para que la detección funcione
independientemente del idioma del correo.

In [ ]:
PATRON_HIDDEN_SPAN = r'visibility:\s*hidden|font-size:\s*0'

mask_hidden = df['text'].astype(str).str.contains(PATRON_HIDDEN_SPAN, regex=True, na=False)
print(f'Filas con técnica de texto oculto (hidden span): {mask_hidden.sum()} ({100*mask_hidden.sum()/len(df):.2f}%)')
print()
print('Por fuente:')
print(df[mask_hidden]['source'].value_counts())
print()
print('Longitud promedio de las filas afectadas vs no afectadas:')
print('Afectadas:', round(df[mask_hidden]['n_caracteres'].mean(), 1))
print('No afectadas:', round(df[~mask_hidden]['n_caracteres'].mean(), 1))

## 5. Idiomas presentes

Detectamos el idioma de cada fila con `langdetect`. Esto NO filtra nada
todavía — solo informa cuántas filas hay por idioma, para decidir el
criterio de filtrado/traducción en el notebook de limpieza.

Como referencia de la literatura (ChatSpamDetector, 2024), datasets de
phishing real de procedencia similar reportan naturalmente entre 12 y 19
idiomas distintos, con inglés dominante pero proporciones significativas
de portugués, alemán, holandés y francés — coherente con lo que se
espera encontrar aquí.

In [ ]:
!pip install langdetect -q

In [ ]:
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

def detectar_idioma_seguro(texto):
    try:
        return detect(str(texto))
    except Exception:
        return 'desconocido'

df['idioma'] = df['text'].apply(detectar_idioma_seguro)

print('Distribución de idiomas (sobre el dataset completo,', len(df), 'filas):')
print(df['idioma'].value_counts())
print()
print(f"% en inglés: {100 * (df['idioma'] == 'en').mean():.1f}%")

## 6. Distribución de idioma por fuente

Verificamos si el multilingüismo se concentra en fuentes específicas.
Se espera que Enron (correspondencia corporativa de EE.UU.) sea
prácticamente 100% inglés, mientras que phishing_pot y Nazario (correos
recolectados de honeypots/bandejas reales a nivel global) concentren
la diversidad de idiomas.

In [ ]:
print('Distribución de idioma por fuente (top 8 idiomas más comunes):')
top_idiomas = df['idioma'].value_counts().head(8).index
print(pd.crosstab(df['source'], df['idioma'])[top_idiomas])

## Decisión registrada (a aplicar en 03_limpieza.ipynb)

Se filtrará el dataset a inglés real (columna `idioma == 'en'`) antes de
la Ronda 1 (entrenar y evaluar en inglés). Esto descarta ~11.5% de las
filas (2,783 de 24,160), concentradas principalmente en `phishing_pot`.

Esta decisión se basa en evidencia de que mezclar idiomas sin tratamiento
adecuado degrada fuertemente el rendimiento del modelo (ver referencia:
Phishing Email Detection Using LLMs, 2026 — aumento de hasta 904% en
falsos positivos al evaluar contra idiomas no vistos en el entrenamiento).

Para la Ronda 2 (traducción a español), se usará la columna `idioma`
generada aquí para traducir cada fila desde SU idioma real (no asumir
que todo es inglés), aprovechando que NLLB-200 soporta 200 idiomas
directamente.

## 7. Duplicados

Exactos (texto idéntico carácter a carácter) y aproximados (mismo
contenido salvo variaciones de mayúsculas/espacios, detectado con una
versión normalizada del texto).

In [ ]:
n_exactos = df.duplicated(subset=['text']).sum()
print(f'Duplicados exactos (texto idéntico): {n_exactos} ({100*n_exactos/len(df):.2f}%)')

# Near-duplicates: normalizamos a minúsculas y espacios simples para
# detectar duplicados que solo difieren en mayúsculas o espacios extra
texto_normalizado = (
    df['text'].astype(str).str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()
)
n_near_dup = texto_normalizado.duplicated().sum()
print(f'Near-duplicates (normalizado, minúsculas/espacios): {n_near_dup} ({100*n_near_dup/len(df):.2f}%)')

In [ ]:
# Desglose de duplicados exactos por fuente: ¿se concentran en alguna
# fuente en particular? (ej. phishing_pot podría tener campañas masivas
# repetidas a muchos destinatarios)
mask_dup = df.duplicated(subset=['text'], keep=False)
print('Duplicados exactos por fuente:')
print(df[mask_dup]['source'].value_counts())

## 7b. Duplicados de plantilla (normalizando campos variables)

Detectamos antes 1,717 duplicados exactos comparando el texto crudo.
Pero muchos correos de phishing usan la MISMA plantilla con valores
variables distintos por destinatario: fecha, monto, ID/código,
hora límite, o el email del destinatario insertado en el cuerpo
(ej. "50 Free Spins... 05-14-2026" vs "...05-08-2026", visto hoy en
varias filas de phishing_pot).

Estos correos NO se detectan como duplicados exactos porque el texto
difiere en esos valores, aunque sean la misma campaña/plantilla.
Normalizamos esos campos a placeholders genéricos para revelar
cuántos duplicados de plantilla existen realmente.

In [ ]:
import re

PATRONES_VARIABLES = {
    'FECHA': r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b|\b(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\.?\s+\d{1,2},?\s+\d{4}\b',
    'MONTO': r'[$€£]\s?[\d,]+\.?\d*|\b\d+[,.]?\d*\s?(USD|EUR|GBP|BTC|ETH)\b',
    'ID_CODIGO': r'\b(ID|Code|Member ID|User ID|Invitation Code)[:\s#]*[A-Za-z0-9-]{4,}\b',
    'HORA_RELATIVA': r'\b\d{1,2}\s?(Hours?|minutes?)\b|before\s+midnight|\d{1,2}:\d{2}\s?(AM|PM|am|pm)?',
    'EMAIL_USUARIO': r'\b[\w.+-]+@[\w-]+\.[\w.-]+\b',
}

texto_normalizado_variables = df['text'].astype(str)
for nombre, patron in PATRONES_VARIABLES.items():
    texto_normalizado_variables = texto_normalizado_variables.str.replace(
        patron, f'<{nombre}>', regex=True, case=False
    )

n_dup_normalizado = texto_normalizado_variables.duplicated().sum()
print(f'Duplicados SIN normalizar variables: {n_exactos} ({100*n_exactos/len(df):.2f}%)')
print(f'Duplicados normalizando fecha/monto/ID/hora/email: {n_dup_normalizado} ({100*n_dup_normalizado/len(df):.2f}%)')
print(f'Duplicados ADICIONALES revelados: {n_dup_normalizado - n_exactos}')

## 8. Resumen comparativo final por fuente

Tabla consolidada con todas las métricas relevantes encontradas en este
EDA, para decidir en `03_limpieza.ipynb` si alguna fuente necesita un
tratamiento de limpieza particular.

In [ ]:
resumen = df.groupby('source').agg(
    n_filas=('text', 'count'),
    longitud_media=('n_caracteres', 'mean'),
    longitud_mediana=('n_caracteres', 'median'),
    longitud_max=('n_caracteres', 'max'),
).round(1)

resumen['pct_no_ingles'] = (
    df[df['idioma'] != 'en'].groupby('source').size() / df.groupby('source').size() * 100
).fillna(0).round(2)

resumen['pct_hidden_span'] = (
    df[mask_hidden].groupby('source').size() / df.groupby('source').size() * 100
).fillna(0).round(2)

resumen['n_duplicados_exactos'] = df[mask_dup].groupby('source').size().fillna(0)

print(resumen)

## 9. HTML residual (más allá del hidden-span)

Ya identificamos la técnica de "hidden text spam" (sección 4), pero esa
es solo una forma específica de HTML. Aquí medimos la presencia general
de marcado HTML sin limpiar (tags `<...>`), que infla artificialmente
la longitud del texto y no aporta señal semántica real para el
clasificador — un transformer entrenado en lenguaje natural no necesita
ver `<div class="...">`, solo el contenido textual.

In [ ]:
PATRON_HTML_TAG = r'<[a-zA-Z][^>]*>'

df['n_tags_html'] = df['text'].astype(str).str.count(PATRON_HTML_TAG)
mask_con_html = df['n_tags_html'] > 0

print(f'Filas con al menos 1 tag HTML: {mask_con_html.sum()} ({100*mask_con_html.sum()/len(df):.2f}%)')
print()
print('Por fuente:')
print(df[mask_con_html]['source'].value_counts())
print()
print('Promedio de tags HTML por fila (solo en filas afectadas):')
print(df[mask_con_html].groupby('source')['n_tags_html'].mean().round(1))

In [ ]:
# Relación entre cantidad de tags HTML y longitud total —
# confirma si el HTML es la causa principal de las filas más largas
print('Correlación entre cantidad de tags HTML y longitud del texto:')
print(df[['n_tags_html', 'n_caracteres']].corr())
print()
print('Top 5 filas con más tags HTML:')
top_html = df.nlargest(5, 'n_tags_html')[['source', 'n_tags_html', 'n_caracteres']]
print(top_html)

## 10. Impacto del truncamiento de tokens (BERT/RoBERTa, límite 512)

Los modelos transformer truncan automáticamente cualquier texto que
exceda su límite de tokens (512 para BERT/RoBERTa base) — el contenido
después de ese punto se descarta silenciosamente, sin error. Medimos
qué porcentaje de filas se vería afectado, usando el tokenizer real del
modelo (no una estimación por caracteres).

In [ ]:
!pip install transformers -q

In [ ]:
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained('bert-base-multilingual-cased')

# Se usa una muestra para no demorar demasiado; ajustar N si se quiere
# medir sobre el dataset completo (más lento, pero exacto)
N_MUESTRA_TOKENS = 2000
muestra_tokens = df.sample(min(N_MUESTRA_TOKENS, len(df)), random_state=42).copy()
muestra_tokens['n_tokens'] = muestra_tokens['text'].apply(
    lambda t: len(tokenizer.tokenize(str(t)))
)

print('Tokens promedio:', round(muestra_tokens['n_tokens'].mean(), 1))
print('Tokens máximo:', muestra_tokens['n_tokens'].max())
print(f"% de filas que superan 512 tokens: {100*(muestra_tokens['n_tokens'] > 512).mean():.1f}%")

In [ ]:
# Cruce: ¿las filas que se truncan son justamente las que tienen HTML
# residual y/o hidden-span? Esto confirmaría que limpiar el HTML antes
# de tokenizar reduce significativamente el truncamiento.
muestra_tokens['tiene_html'] = muestra_tokens['n_tags_html'] > 0
muestra_tokens['se_trunca'] = muestra_tokens['n_tokens'] > 512

print('Tasa de truncamiento según presencia de HTML:')
print(muestra_tokens.groupby('tiene_html')['se_trunca'].mean().round(3) * 100, '%')
print()
print('Por fuente, % de filas que se truncan:')
print(muestra_tokens.groupby('source')['se_trunca'].mean().round(3) * 100, '%')

## Conclusiones del EDA — decisiones a aplicar en 03_limpieza.ipynb

**Hallazgos principales:**
1. Balance de clases: 50/50 (12,080 phishing vs 12,080 legítimo) — correcto desde la extracción.
2. 11.5% de las filas no están en inglés (concentrado en phishing_pot: 28.98%).
3. 8.17% usa la técnica de "hidden text spam" (HTML con texto invisible).
4. 35.44% tiene HTML residual sin limpiar, con correlación de 0.65 con la longitud total.
5. 7.11% son duplicados exactos; +112 adicionales si se normalizan campos variables (fecha/monto/ID).
6. **37.8% de las filas supera el límite de 512 tokens y se trunca al entrenar.**
7. **El HTML residual es la causa principal del truncamiento**: 77.4% de truncamiento en filas con HTML, vs 15.1% sin HTML.
8. `phishing_pot` es consistentemente la fuente más problemática en todas las dimensiones medidas.
9. Encoding roto y ofuscación de caracteres (no cuantificado todavía con un conteo exacto en este EDA, pero confirmado cualitativamente en sesiones de limpieza previas sobre datos de esta misma procedencia): homóglifos cirílicos/griegos/armenios, marcas combinadoras invisibles entre letras, y entidades HTML corrompidas por el propio marcador de anonimización `phishing@pot` insertado dentro del código numérico de la entidad.

**Decisiones para 03_limpieza.ipynb (en orden de ejecución, no solo de prioridad):**
1. **Corregir encoding** (ftfy) y normalizar homóglifos/marcas invisibles — debe ir primero, porque corrige la base de texto sobre la que actúan todos los pasos siguientes; si se limpia HTML antes sobre texto con encoding roto, los regex pueden fallar o dejar residuos.
2. **Eliminar HTML residual** (tags + hidden-span) — máxima prioridad de impacto, reduce el truncamiento de tokens más que cualquier otro paso.
3. **Filtrar a inglés real** (columna `idioma == 'en'`) para la Ronda 1 de entrenamiento.
4. **Eliminar duplicados exactos**, normalizando antes fecha/monto/ID/hora/email.
5. **Re-evaluar el balance de clases** después de los pasos anteriores, dado que reducirán principalmente `phishing_pot`.